In [2]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.label.label_generator import LabelGenerator

lg = LabelGenerator(time_start=1430, time_end=1457, price_type="twap")
price_path, label_path = lg.generate_and_save()
print(f"价格表: {price_path}")
print(f"Label:  {label_path}")

LabelGenerator [TWAP_1430_1457]:   0%|          | 0/1261 [00:00<?, ?day/s]

价格表: /root/autodl-tmp/Strategy/outputs/labels/TWAP_1430_1457.fea
Label:  /root/autodl-tmp/Strategy/outputs/labels/LABEL_TWAP_1430_1457.fea


In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy import config
from Strategy.factor.daily_factor_library import DailyFactorLibraryAdapter

# 日频因子需覆盖 训练+验证+样本外 至数据末，否则 val / oos / 打分缺特征（见 config 注释）
# end_date 不传 = 用 Daily_data 中最后交易日；仅调试可写 end_date=config.TRAIN_END
adapter = DailyFactorLibraryAdapter()
saved = adapter.compute_and_save_all(
    start_date=config.TRAIN_START,
    # end_date=config.TRAIN_END,  # 仅训练内调试时取消注释
)
print(f"已保存 {len(saved)} 个因子")

In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy.factor.factor_base import FactorRegistry
import Strategy.factor.minute_derived_factors   # 触发注册
import Strategy.factor.custom_factors           # 触发注册
 
FactorRegistry.compute_all()

In [ ]:
import sys, logging
sys.path.insert(0, '/root/autodl-tmp')
logging.basicConfig(level=logging.INFO)

from Strategy.factor.minute_derived_factors import JumpVariationFactor

jv = JumpVariationFactor()
jv.compute_and_save()  # 输出 RV / RVC / RVJ 等 13 个因子

In [5]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel

# 加载
factor_dict = load_all_factors()          # 自动读取 outputs/factors/ 下所有 .fea
label_df    = load_label("TWAP_1430_1457")

# 拼接长表 Panel
panel = build_panel(factor_dict, label_df)
print(panel.shape)   # (样本数, 因子数+3)

# 按时间划分 (config.py 中已定义日期)
# Train: 2021-01-01 ~ 2023-08-01
# Val:   2023-09-01 ~ 2024-09-01
# OOS:   2024-09-01 之后
train, val, oos = split_panel(panel)

(6660324, 137)


In [6]:
from Strategy.model.trainer import XGBTrainer

trainer = XGBTrainer(
    params={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 50,
        "tree_method": "hist",  # XGBoost 2+ 统一用 hist；GPU 时配合 device 见下行
        "device": "cuda",  # 无独显或报 CUDA 错时改为 "cpu" 或删此键
        "verbosity": 0,
    },
    num_boost_round=500,
    early_stopping_rounds=50,
)

trainer.train(train, val)
trainer.save_model()   # 保存到 outputs/scores/xgb_model.pkl


libgomp: Invalid value for environment variable OMP_NUM_THREADS


[0]	train-rmse:0.02893	val-rmse:0.02977
[100]	train-rmse:0.02865	val-rmse:0.02951
[200]	train-rmse:0.02853	val-rmse:0.02943
[300]	train-rmse:0.02843	val-rmse:0.02940
[400]	train-rmse:0.02835	val-rmse:0.02938
[499]	train-rmse:0.02828	val-rmse:0.02936


PosixPath('/root/autodl-tmp/Strategy/outputs/scores/xgb_model.pkl')

In [7]:
from Strategy.model.scorer import generate_scores, load_scores
from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label

factor_dict = load_all_factors()
label_df    = load_label("TWAP_1430_1457")

score_path = generate_scores(
    trainer=trainer,
    factor_dict=factor_dict,
    label_df=label_df,
    model_name="xgb",
    label_tag="TWAP_1430_1457",
    normalize=True,               # 每日截面 Z-Score 标准化
)
print(f"打分已保存: {score_path}")

# 后续加载
score_df = load_scores("xgb", "TWAP_1430_1457")

打分已保存: /root/autodl-tmp/Strategy/outputs/scores/SCORE_xgb_TWAP_1430_1457.fea


In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

from Strategy.backtest.quick_backtest import run_quick_backtest
from Strategy.model.scorer import load_scores
from Strategy.label.label_generator import load_label
from Strategy import config

score_df = load_scores("xgb", "TWAP_1430_1457")
label_df = load_label("TWAP_1430_1457")

# 可自行调整想看的 topN 曲线, 例如 (10, 20, 50, 100, 200)
TOP_KS = (20, 50, 100)

# ⚠️ 必须指定 start_date=config.VAL_START (或 OOS_START), 否则训练期的
# 样本内预测会拉高 group1 收益, 产生虚假的"高收益"假象。
quick_ret = run_quick_backtest(
    score_df=score_df,
    label_df=label_df,
    n_groups=20,
    output_dir=config.BT_RESULT_DIR,
    start_date=config.VAL_START,   # 从验证集起计, 排除训练期样本内日期
    top_ks=TOP_KS,
)

In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp')

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from Strategy.backtest.event_backtest import BacktestRunner
from Strategy.model.scorer import load_scores
from Strategy import config

score_df = load_scores("xgb", "TWAP_1430_1457")

# 可自行调整想看的每日买入 topN, 例如 (10, 20, 50, 100, 200)
TOP_NS = (20, 50, 100)
event_results = {}

for top_n in TOP_NS:
    runner = BacktestRunner(
        score_df=score_df,
        top_n=top_n,
        n_quantile_groups=20,
        rebalance_freq=1,       # 每日调仓, 即每日只买入当日 topN
        initial_capital=10_000_000.0,
        frictionless=True,      # 与 run_quick_backtest 无佣金/印花税假设对齐; 实盘改 False
    )
    result = runner.run(start_date=config.VAL_START, end_date=None)
    result.plot(save_dir=config.BT_RESULT_DIR)
    result.save_details(config.BT_RESULT_DIR)
    event_results[f"top{top_n}"] = result.nav_df.set_index("TRADE_DATE")["nav"] / result.initial_capital

# 汇总画一张 top20/top50/top100 事件回测净值对比图
nav_compare = pd.DataFrame(event_results)
fig, ax = plt.subplots(figsize=(13, 5))
for col in nav_compare.columns:
    ax.plot(nav_compare.index, nav_compare[col], label=col, lw=1.2)
ax.axvline(pd.Timestamp(config.OOS_START), color="black", ls="--", lw=0.8, alpha=0.75)
ax.axhline(1.0, color="gray", lw=0.5)
ax.set_title("Event Backtest TopN NAV Compare")
ax.set_ylabel("Net Value")
ax.legend()
plt.xticks(rotation=45, fontsize=7)
plt.tight_layout()
compare_path = config.BT_RESULT_DIR / "event_topn_nav_compare.png"
fig.savefig(compare_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"TopN 对比图已保存: {compare_path}")
nav_compare.tail()

Error.  nthreads must be a positive integerOMP: Warning #80: OMP_NUM_THREADS="0": value too small.
OMP: Info #104: OMP_NUM_THREADS value "1" will be used.
